In [1]:
# Cell 1: imports and setup

import sys
from pathlib import Path

# Add src/ to Python path so we can import our modules
notebook_dir = Path.cwd()
project_root = notebook_dir.parent  # assignment_2/
sys.path.insert(0, str(project_root / "src"))

# Verify paths
print("Notebook dir:", notebook_dir)
print("Project root:", project_root)
print("Data dir:    ", project_root / "data" / "raw")


Notebook dir: /Users/francescoorma99/Documents/GitHub/RES-Assignments/assignment_2/notebooks
Project root: /Users/francescoorma99/Documents/GitHub/RES-Assignments/assignment_2
Data dir:     /Users/francescoorma99/Documents/GitHub/RES-Assignments/assignment_2/data/raw


In [4]:
# Cell 2: load raw data and sanity-check

from data_loader import load_all

raw = load_all(data_dir=project_root / "data" / "raw")

print("Wind capacity factor:")
print(f"  shape: {raw.wind_cf.shape}, range: [{raw.wind_cf.min():.3f}, {raw.wind_cf.max():.3f}], mean: {raw.wind_cf.mean():.3f}")
print(f"  index: {raw.wind_cf.index[0]} → {raw.wind_cf.index[-1]}")
print()

print("DA prices (EUR/MWh):")
print(f"  shape: {raw.da_prices.shape}, range: [{raw.da_prices.min():.2f}, {raw.da_prices.max():.2f}], mean: {raw.da_prices.mean():.2f}")
print(f"  index: {raw.da_prices.index[0]} → {raw.da_prices.index[-1]}")
print()

print("Imbalance:")
print(f"  shape: {raw.imbalance.shape}")
print(f"  columns: {list(raw.imbalance.columns)}")
print(f"  index: {raw.imbalance.index[0]} → {raw.imbalance.index[-1]}")
print()
print(raw.imbalance.describe())

Wind capacity factor:
  shape: (8760,), range: [0.000, 0.939], mean: 0.265
  index: 2024-01-01 00:00:00+00:00 → 2024-12-30 23:00:00+00:00

DA prices (EUR/MWh):
  shape: (8760,), range: [-59.96, 936.31], mean: 70.97
  index: 2024-01-01 00:00:00 → 2024-12-30 23:00:00

Imbalance:
  shape: (8760, 3)
  columns: ['mFRRUpActBal', 'mFRRDownActBal', 'ImbalanceMWh']
  index: 2024-01-01 00:00:00 → 2024-12-30 23:00:00

       mFRRUpActBal  mFRRDownActBal  ImbalanceMWh
count   8760.000000     8760.000000   8760.000000
mean       5.860360        9.814498   -121.936998
std       27.773364       29.500083    215.530390
min        0.000000        0.000000  -1265.000000
25%        0.000000        0.000000   -220.749996
50%        0.000000        0.000000   -105.649997
75%        0.000000        0.000000    -18.200001
max      537.000000      349.250000   1128.599976


In [5]:
# Quick estimate of p_deficit from ImbalanceMWh
imb = raw.imbalance["ImbalanceMWh"]

n_deficit = (imb < 0).sum()
n_surplus = (imb > 0).sum()
n_zero    = (imb == 0).sum()

print(f"Deficit hours (Imb < 0): {n_deficit:5d}  ({n_deficit/8760:.1%})")
print(f"Surplus hours (Imb > 0): {n_surplus:5d}  ({n_surplus/8760:.1%})")
print(f"Zero hours (Imb == 0):   {n_zero:5d}  ({n_zero/8760:.1%})")
print()
print(f"Estimated Bernoulli p (P(deficit)) = {n_deficit/8760:.3f}")

Deficit hours (Imb < 0):  6932  (79.1%)
Surplus hours (Imb > 0):  1812  (20.7%)
Zero hours (Imb == 0):      16  (0.2%)

Estimated Bernoulli p (P(deficit)) = 0.791


In [6]:
# Verify the sign convention of ImbalanceMWh by cross-checking with mFRR activations
imb = raw.imbalance

deficit_hours = imb[imb["mFRRUpActBal"] > 0]["ImbalanceMWh"]
surplus_hours = imb[imb["mFRRDownActBal"] > 0]["ImbalanceMWh"]

print("Hours with mFRR Up activated (system in deficit):")
print(f"  count: {len(deficit_hours)}")
print(f"  ImbalanceMWh — mean: {deficit_hours.mean():.1f}, median: {deficit_hours.median():.1f}")
print(f"  Fraction with ImbalanceMWh < 0: {(deficit_hours < 0).mean():.1%}")
print()
print("Hours with mFRR Down activated (system in surplus):")
print(f"  count: {len(surplus_hours)}")
print(f"  ImbalanceMWh — mean: {surplus_hours.mean():.1f}, median: {surplus_hours.median():.1f}")
print(f"  Fraction with ImbalanceMWh > 0: {(surplus_hours > 0).mean():.1%}")

Hours with mFRR Up activated (system in deficit):
  count: 1062
  ImbalanceMWh — mean: -92.2, median: -70.7
  Fraction with ImbalanceMWh < 0: 73.3%

Hours with mFRR Down activated (system in surplus):
  count: 2029
  ImbalanceMWh — mean: -116.8, median: -109.2
  Fraction with ImbalanceMWh > 0: 22.1%


In [7]:
# Estimate Bernoulli p using only hours with mFRR activations
imb = raw.imbalance

mask_up = imb["mFRRUpActBal"] > 0     # system in deficit
mask_dn = imb["mFRRDownActBal"] > 0    # system in surplus

# Sanity check: are any hours BOTH up and down activated?
both = (mask_up & mask_dn).sum()
print(f"Hours with both Up AND Down activated: {both}")

n_deficit = mask_up.sum()
n_surplus = mask_dn.sum()
n_active = n_deficit + n_surplus

p = n_deficit / n_active
print(f"Deficit hours (mFRR Up > 0):   {n_deficit}")
print(f"Surplus hours (mFRR Down > 0): {n_surplus}")
print(f"Total active hours:            {n_active}  ({n_active/8760:.1%} of year)")
print(f"\nEstimated Bernoulli p (P(deficit | active)) = {p:.3f}")

Hours with both Up AND Down activated: 30
Deficit hours (mFRR Up > 0):   1062
Surplus hours (mFRR Down > 0): 2029
Total active hours:            3091  (35.3% of year)

Estimated Bernoulli p (P(deficit | active)) = 0.344
